# RETFound Encoding v3 — EyePACS + Messidor + OLIVES

**Before running:** Runtime → Change runtime type → T4 GPU

**Run all cells top to bottom. Every session starts at Cell 0.**

### What this does
1. Downloads EyePACS from Kaggle (~33 GB)
2. Extracts all images (~33 GB) — needs ~69 GB free total
3. Encodes all images with RETFound (T4 GPU, ~15 min)
4. Encodes Messidor-2 (you upload IMAGES4.zip, ~241 MB)
5. Encodes OLIVES streamed from HuggingFace (no upload needed)
6. Saves all embeddings to Google Drive

In [ ]:
# ── Cell 0: Setup — run every fresh session ────────────────────────────────
!pip install timm kaggle datasets -q
!apt-get install -q p7zip-full

import os, shutil, subprocess, zipfile, glob
import numpy as np
import torch
import timm
import pandas as pd
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from pathlib import Path
from google.colab import drive, files

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Check GPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected! Switch to T4 GPU runtime before continuing.')

# Mount Google Drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/synapse/embeddings'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Drive mounted. Embeddings → {OUT_DIR}')

# Kaggle credentials
print('\nUpload kaggle.json:')
uploaded = files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle ready.')

# Disk check
_, _, free = shutil.disk_usage('/')
print(f'Free disk: {free//1e9:.1f} GB (need ~69 GB)')
if free < 65e9:
    print('WARNING: Low disk space. Factory reset runtime may help: Runtime → Factory reset runtime.')
else:
    print('Disk space OK.')

In [ ]:
# ── Cell 2: Download EyePACS from Kaggle (skips if already present) ────────
EYEPACS = '/content/eyepacs'
os.makedirs(EYEPACS, exist_ok=True)

# Labels
if not os.path.exists(f'{EYEPACS}/trainLabels.csv'):
    print('Downloading labels...')
    !kaggle competitions download -c diabetic-retinopathy-detection -f trainLabels.csv.zip -p /content/eyepacs/
    !cd /content/eyepacs && unzip -q trainLabels.csv.zip
else:
    print('trainLabels.csv already present — skipping.')

# Image chunks
for i in range(1, 6):
    outer = f'{EYEPACS}/train.zip.00{i}.zip'
    inner = f'/content/eyepacs/inner/train.zip.00{i}'
    if os.path.exists(outer):
        print(f'train.zip.00{i}.zip already present — skipping download.')
    elif os.path.exists(inner):
        print(f'train.zip.00{i} (inner part) already extracted — skipping download.')
    else:
        print(f'Downloading chunk {i}/5...')
        !kaggle competitions download -c diabetic-retinopathy-detection -f train.zip.00{i} -p /content/eyepacs/

print('\nFiles present:')
for f in sorted(os.listdir(EYEPACS)):
    size = os.path.getsize(f'{EYEPACS}/{f}')
    print(f'  {f}: {size/1e9:.1f} GB')
_, _, free = shutil.disk_usage('/')
print(f'Free: {free//1e9:.1f} GB')

In [ ]:
# ── Cell 1: Load RETFound ──────────────────────────────────────────────────
print('Loading RETFound (bitfount/RETFound_MAE)...')
model = timm.create_model('hf_hub:bitfount/RETFound_MAE', pretrained=True, num_classes=0)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad = False

_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

with torch.no_grad():
    DIM = model(torch.zeros(1, 3, 224, 224).to(DEVICE)).shape[-1]
print(f'RETFound loaded. Embedding dim: {DIM}')

In [ ]:
# ── Cell 3: Extract outer zips → inner split parts (skips if already done) ─
INNER = '/content/eyepacs/inner'
os.makedirs(INNER, exist_ok=True)

for i in range(1, 6):
    outer = f'{EYEPACS}/train.zip.00{i}.zip'
    inner = f'{INNER}/train.zip.00{i}'
    if os.path.exists(inner):
        print(f'train.zip.00{i} already extracted — skipping.')
        if os.path.exists(outer): os.remove(outer)
        continue
    if not os.path.exists(outer):
        print(f'MISSING: {outer} — re-run Cell 2')
        continue
    print(f'Extracting chunk {i}...')
    with zipfile.ZipFile(outer) as zf:
        zf.extractall(INNER)
    os.remove(outer)
    _, _, free = shutil.disk_usage('/')
    print(f'  Done. Free: {free//1e9:.1f} GB')

parts = sorted(glob.glob(f'{INNER}/train.zip.*'))
print(f'\nInner parts ready ({len(parts)}): {[os.path.basename(p) for p in parts]}')
_, _, free = shutil.disk_usage('/')
print(f'Free disk: {free//1e9:.1f} GB')

In [ ]:
# ── Cell 4: Extract all images from split archive (skips if already done) ──
IMAGES = '/content/eyepacs/images'
os.makedirs(IMAGES, exist_ok=True)

existing = glob.glob(f'{IMAGES}/*.jpeg')
if len(existing) > 30000:
    print(f'Images already extracted: {len(existing)} — skipping.')
else:
    _, _, free = shutil.disk_usage('/')
    print(f'Free before extraction: {free//1e9:.1f} GB')
    print('Extracting all images with 7-zip...')
    result = subprocess.run(
        ['7z', 'e', f'{INNER}/train.zip.001', f'-o{IMAGES}', '-y', '-bd'],
        capture_output=True, text=True
    )
    print(f'7-zip exit code: {result.returncode}')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    shutil.rmtree(INNER)

imgs = glob.glob(f'{IMAGES}/*.jpeg')
_, _, free = shutil.disk_usage('/')
print(f'Images ready: {len(imgs)}')
print(f'Free disk: {free//1e9:.1f} GB')

In [ ]:
# ── Cell 5: Encode EyePACS ─────────────────────────────────────────────────
from torchvision import transforms
from PIL import Image
import os, shutil, glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd

# Define transform locally so DataLoader workers can access it
_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class EyePACSDataset(Dataset):
    def __init__(self, img_dir, csv_path):
        df = pd.read_csv(csv_path)
        df['path'] = df['image'].apply(lambda x: os.path.join(img_dir, f'{x}.jpeg'))
        self.df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
        print(f'EyePACS dataset: {len(self.df)} images found')
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['path']).convert('RGB')
        return _TRANSFORM(img), int(row['level']), str(row['image'])

@torch.no_grad()
def encode_dataset(dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    embs, labels, names = [], [], []
    for batch in tqdm(loader, desc='Encoding'):
        imgs = batch[0].to(DEVICE)
        embs.append(model(imgs).cpu().numpy())
        labels.extend(batch[1].numpy().tolist())
        names.extend(batch[2])
    return np.concatenate(embs, axis=0), np.array(labels), names

eyepacs = EyePACSDataset(IMAGES, f'{EYEPACS}/trainLabels.csv')
embs_e, lbls_e, names_e = encode_dataset(eyepacs)

np.save(f'{OUT_DIR}/eyepacs_retfound.npy', embs_e)
np.save(f'{OUT_DIR}/eyepacs_labels.npy', lbls_e)
print(f'EyePACS embeddings: {embs_e.shape}')

shutil.rmtree(IMAGES)
_, _, free = shutil.disk_usage('/')
print(f'Free after cleanup: {free//1e9:.1f} GB')

In [ ]:
# ── Cell 6: Encode Messidor ────────────────────────────────────────────────
# Upload IMAGES4.zip from your laptop: data/raw/messidor/IMAGES4.zip (~241 MB)

print('Upload IMAGES4.zip (data/raw/messidor/IMAGES4.zip on your laptop):')
uploaded = files.upload()

MESSIDOR = '/content/messidor'
os.makedirs(MESSIDOR, exist_ok=True)
for fname in uploaded:
    subprocess.run(['7z', 'e', fname, f'-o{MESSIDOR}', '-y', '-bd'], capture_output=True)
    os.remove(fname)

class FolderDataset(Dataset):
    def __init__(self, folder):
        self.paths = sorted(
            p for p in Path(folder).rglob('*')
            if p.suffix.lower() in ['.png', '.jpg', '.jpeg']
        )
        print(f'Messidor images: {len(self.paths)}')
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return _TRANSFORM(Image.open(self.paths[i]).convert('RGB')), 0

@torch.no_grad()
def encode_folder(dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    embs = []
    for imgs, _ in tqdm(loader, desc='Encoding Messidor'):
        embs.append(model(imgs.to(DEVICE)).cpu().numpy())
    return np.concatenate(embs, axis=0)

messidor_ds = FolderDataset(MESSIDOR)
embs_m = encode_folder(messidor_ds)
np.save(f'{OUT_DIR}/messidor_retfound.npy', embs_m)
print(f'Messidor embeddings: {embs_m.shape}')
shutil.rmtree(MESSIDOR)

In [ ]:
# ── Cell 7: Encode OLIVES (streamed from HuggingFace) ─────────────────────
# Streams images directly — no 31 GB download needed.
# Key is 'Image' (capital I) not 'image'.

from datasets import load_dataset
import PIL

print('Streaming OLIVES from HuggingFace...')
ds = load_dataset('gOLIVES/OLIVES_Dataset', 'disease_classification',
                  split='train', streaming=True)

embs_o, eye_ids, lbls_o = [], [], []
buf_imgs, buf_eyes, buf_lbls = [], [], []
BATCH = 64
n = 0

def flush():
    if not buf_imgs: return
    with torch.no_grad():
        embs_o.append(model(torch.stack(buf_imgs).to(DEVICE)).cpu().numpy())
    eye_ids.extend(buf_eyes)
    lbls_o.extend(buf_lbls)
    buf_imgs.clear(); buf_eyes.clear(); buf_lbls.clear()

for sample in ds:
    try:
        img = sample['Image']  # capital I
        if not isinstance(img, PIL.Image.Image):
            img = PIL.Image.fromarray(img)
        buf_imgs.append(_TRANSFORM(img.convert('RGB')))
        buf_eyes.append(sample.get('Eye_ID', -1))
        buf_lbls.append(sample.get('Disease Label', -1))
        n += 1
        if len(buf_imgs) >= BATCH: flush()
        if n % 1000 == 0: print(f'  {n} images...')
    except Exception as e:
        print(f'Error at sample {n}: {e}')
        break

flush()

if embs_o:
    embs_o = np.concatenate(embs_o, axis=0)
    np.save(f'{OUT_DIR}/olives_retfound.npy', embs_o)
    np.save(f'{OUT_DIR}/olives_eye_ids.npy', np.array(eye_ids))
    np.save(f'{OUT_DIR}/olives_labels.npy', np.array(lbls_o))
    print(f'OLIVES embeddings: {embs_o.shape}, unique eyes: {len(np.unique(eye_ids))}')
else:
    print('ERROR: No embeddings collected.')

In [ ]:
# ── Cell 8: Validate everything ────────────────────────────────────────────
print('=== Validation ===')
expected = ['eyepacs_retfound.npy', 'eyepacs_labels.npy',
            'messidor_retfound.npy', 'olives_retfound.npy',
            'olives_eye_ids.npy', 'olives_labels.npy']
all_good = True
for fname in expected:
    fpath = f'{OUT_DIR}/{fname}'
    if os.path.exists(fpath):
        arr = np.load(fpath)
        print(f'  {fname}: shape={arr.shape}, mean={arr.mean():.4f} ✓')
    else:
        print(f'  {fname}: MISSING ✗')
        all_good = False

if all_good:
    print('\nAll embeddings saved to Google Drive at: My Drive/synapse/embeddings/')
    print('Download them and place in: data/processed/embeddings/ on your laptop.')
else:
    print('\nSome files missing — check errors in the cells above.')